In [2]:
import csv
import pickle
import pandas as pd
import numpy as np

import quartz.grids as qg
from quartz.data import MoleculeData
from quartz.dataprocessing import assign_props, calculate_overlap, get_overlap_grid
from quartz.grids import create_grid
from quartz.io import load_and_process

In [3]:
# mols = load_and_process(input_dir="tests/sample_data/", size=0.1)

In [4]:
# from pathlib import Path
# from quartz.xtbinterface import generate_input, get_props
# from quartz.utils import get_mol, xyz_to_mol
# from rdkit.Chem.rdmolfiles import MolFromPDBFile
# from rdkit.Chem.Descriptors import NumRadicalElectrons
# from rdkit.Chem.rdmolops import GetFormalCharge

# path = Path("tests/sample_data/proteins/")
# file = "1EZQ_dock_reduced.pdb"

# rdmol = MolFromPDBFile(path / file)
# chrg = GetFormalCharge(rdmol)
# uhf = NumRadicalElectrons(rdmol)
# print(chrg, uhf)

# generate_input(path, 1.0, chrg, uhf)
# mol_1EZQ = get_mol(str(path / file))
# res = get_props(str(path / file))

# docked_1EZQ = MoleculeData(
#     mol_1EZQ.atomic_numbers,
#     mol_1EZQ.geometry,
#     res,
#     "AA",
#     xyz_to_mol(str(path  / "1EZQ_docked.pdb")),
#     path.name,
# )
# # docked_1EZQ = load_and_process(input_dir="tests/sample_data/docked_ligands/1EZQ_docked.pdb")

In [5]:
# docked_1EZQ

In [6]:
# pickle.dump(mols, open("FX01.pkl", 'wb'))

In [7]:
with open("./tests/sample_data/FX01.pkl", "rb") as f:
    mols = pickle.load(f)

In [8]:
# mols[0].res.keys()

In [9]:
with open("tests/sample_data/FX_data.csv", "r", encoding="utf-8-sig") as f:
    csvreader = csv.DictReader(f)
    for row in csvreader:
        # print(row)
        for mol in mols:
            if mol.filename[0:4] is not None and mol.filename[0:4] == row["PDB ID "]:
                mol.pdb_code = row["PDB ID "]
                mol.ki = float(row["Ki (nM)"])
                mol.pki = float(row["pKi "])
                # print(f"Assigned PDB code {row['PDB ID ']} to {mol.filename}")
                break

mols.sort(key=lambda x: x.ki)

In [10]:
# from rdkit.Chem.Draw import IPythonConsole, MolsToGridImage

# MolsToGridImage([mol.rdmol for mol in mols], legends=[mol.pdb_code for mol in mols], subImgSize=(300, 200))


In [11]:
for i in mols:
    assign_props(
        i,
        [
            "coordination number",
            "dispersion coef",
            "polarizibility",
            "charges",
            "fullpop",
            "fod_s",
            "fod_p",
            "fod_d",
            "fukui+",
            "fukui-",
            "fukui0",
        ],
    )

In [12]:
# np.min(np.abs(mols[3].res['coordination number']))

In [13]:
grids = qg.create_grid(
    mols,
    props=[
        "coordination number",
        "dispersion coef",
        "polarizibility",
        "charges",
        "fullpop",
        "fod_s",
        "fod_p",
        "fod_d",
        "fukui+",
        "fukui-",
        "fukui0",
    ],
    padding=5.0,
    size=0.3,
    threshold={
        "dispersion coef": 0.1,
        "coordination number": 0.01,
        "fukui+": 0.001,
        "fukui-": 0.001,
        "fukui0": 0.001,
        "polarizibility": 0.01,
        "charges": 0.001,
        "fullpop": 0.01,
        "fod_s": 0.01,
        "fod_p": 0.01,
        "fod_d": 0.01,
    },
    clamp_dist=0.5,
    clamp_falloff=1000,
)

Grid lengths:  35.0 35.0 44.0
Origin:  -2.72099029 61.69410415 92.5117578
Cell count:  117 117 147
Processing:  COc1cc(Cl)cc(C(=O)Nc2ccc(Cl)cn2)c1NC(=O)c1scc(CN(C)c2ncco2)c1Cl
Processing:  C[NH+](C)Cc1nccn1-c1ccc(NC(=O)c2cc(C(F)(F)F)nn2-c2ccc3onc(N)c3c2)c(F)c1
Processing:  COC(=O)[C@H](Cc1cccc(C(N)=[NH2+])c1)[C@@H](C)NC(=O)c1ccc(C2=CCN(O)C=C2)cc1
Processing:  COC(=O)[C@H](Cc1cccc(C(N)=[NH2+])c1)[C@@H](C)NC(=O)c1ccc(-c2cccc(C[NH3+])c2)cc1
Processing:  C[N@H+]1CC[N@H+](Cc2csc(C(=O)Nc3ccc(Cl)cc3C(=O)Nc3ccc(Cl)cc3)c2Cl)CC1
Processing:  NC(=[NH2+])c1ccc(CN2CCN(S(=O)(=O)c3cc4ccc(Cl)cc4s3)CC2=O)cc1
Processing:  O=C1CN(S(=O)(=O)c2cc3ccc(Cl)cc3s2)CCN1Cc1cc2cnccc2n1CCO
Processing:  O=C1[C@@H](NS(=O)(=O)c2cc3ncccc3s2)CCN1CC1=CC2=CCN=CC2=N1
Processing:  NC(=[NH2+])c1cccc(CN2CCN(S(=O)(=O)c3cc4ccc(Cl)cc4s3)CC2=O)c1
Processing:  Nc1nccc2ccc(CN3CC[C@H](NS(=O)(=O)c4cc5ncccc5s4)C3=O)cc12
Processing:  C[N@@H+]1CCc2nc(C(=O)N3CCN(S(=O)(=O)c4ccc5cc(Cl)ccc5c4)C[C@@H]3C(N)=O)sc2C1
Processing:  Cc1nc2ccccc2n1C

In [14]:
# grids = pickle.load(open("tests/sample_data/FX_grids_05.pkl", 'rb'))

In [20]:
pickle.dump(grids, open("tests/sample_data/FX_grids_03.pkl", "wb"))

In [16]:
overlaps = []
for i in range(2, len(mols)):
    overlaps.append(
        calculate_overlap(
            grids[0:i],
            [
                "coordination number",
                "dispersion coef",
                "polarizibility",
                "charges",
                "fullpop",
                "fod_s",
                "fod_p",
                "fod_d",
                "fukui+",
                "fukui-",
                "fukui0",
            ],
        )
    )

In [17]:
[
    print(f"Overlap for grids 0 to {i}: ", float(dic["fukui-"]))
    for i, dic in enumerate(overlaps, 2)
]

Overlap for grids 0 to 2:  0.2285
Overlap for grids 0 to 3:  0.6052
Overlap for grids 0 to 4:  0.698
Overlap for grids 0 to 5:  0.7601
Overlap for grids 0 to 6:  0.7969
Overlap for grids 0 to 7:  0.8343
Overlap for grids 0 to 8:  0.8582
Overlap for grids 0 to 9:  0.8798
Overlap for grids 0 to 10:  0.8936
Overlap for grids 0 to 11:  0.9072
Overlap for grids 0 to 12:  0.9139
Overlap for grids 0 to 13:  0.9174


[None, None, None, None, None, None, None, None, None, None, None, None]

In [18]:
fukuim_overlap_05 = get_overlap_grid(grids[0:5], ["fukui-"], save_dx=True)

In [19]:
raise Exception()

Exception: 

In [ ]:
print(f"Grid origin: {grids[0].grid_props[1]}")
print(f"Grid shape: {[grids[0].npGrid[key].shape for key in grids[0].npGrid.keys()]}")
print(f"Grid spacing: {grids[0].grid_props[2]}")

Grid origin: [np.float64(66.69410415), np.float64(91.25865405)]
Grid shape: [(70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88), (70, 70, 88)]
Grid spacing: [np.float64(97.5117578), np.float64(130.75959925)]


In [ ]:
qg.to_dxgrid(grids)

In [ ]:
for key in grids[0].npGrid.keys():
    print(f"{key}:")
    print([max(x.res[key]) for x in mols[0:5]])
    print([np.max(grid.npGrid[key]) for grid in grids])

coordination number:
[3.745, 3.735, 3.839, 3.84, 3.745]
[np.float16(4.31), np.float16(4.285), np.float16(4.246), np.float16(4.258), np.float16(4.51), np.float16(4.16), np.float16(4.562), np.float16(4.586), np.float16(4.3), np.float16(4.4), np.float16(4.65), np.float16(4.312), np.float16(4.367), np.float16(4.438)]
dispersion coef:
[137.131, 29.917, 29.888, 29.344, 132.595]
[np.float16(137.2), np.float16(30.73), np.float16(30.88), np.float16(30.5), np.float16(132.9), np.float16(138.8), np.float16(138.4), np.float16(138.8), np.float16(138.2), np.float16(139.4), np.float16(135.9), np.float16(30.84), np.float16(31.47), np.float16(31.97)]
polarizibility:
[19.709, 8.972, 8.964, 8.883, 19.381]
[np.float16(19.77), np.float16(10.96), np.float16(10.42), np.float16(10.31), np.float16(19.44), np.float16(19.88), np.float16(19.84), np.float16(19.88), np.float16(19.83), np.float16(19.9), np.float16(19.67), np.float16(10.336), np.float16(11.09), np.float16(11.445)]
charges:
[0.26705973, 0.50402464, 0.3

In [ ]:
df = pd.DataFrame([x.__dict__ for x in mols])

In [ ]:
df["pki"] = pd.to_numeric(df["pki"])
df.sort_values("pki", inplace=True, ascending=False)